# Skin Disease Classifier - Colab Training

Trains a transfer-learning CNN on the **22-class skin disease image dataset** entirely on Colab.
The dataset (`pacificrm/skindiseasedataset`) is downloaded straight to the Colab VM via **kagglehub** - **nothing is stored locally on your machine.**

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.

---
### Workflow
1. Check GPU
2. Pull in the project code (`src/`)
3. Install dependencies
4. Authenticate to Kaggle & download the dataset (kagglehub)
5. Configure & train
6. Evaluate (accuracy, F1, confusion matrix)
7. Save the trained model to Google Drive

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Get the project code

**Option A (recommended) - clone from GitHub.** Set `REPO_URL` below.

**Option B - upload manually.** Zip the project's `src/`, `configs/`, `scripts/` folders, upload via the Files panel, unzip into `/content/skin-disease`, then set `USE_GITHUB = False`.

In [ ]:
USE_GITHUB = True  # set False if you uploaded the code manually
REPO_URL = 'https://github.com/Ammar8065/Skin-Disease-Classification.git'
PROJECT_DIR = '/content/skin-disease'

import os, sys, shutil, subprocess

if USE_GITHUB:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    # Run clone via subprocess (more reliable than !git over the VS Code bridge).
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, PROJECT_DIR],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0 or not os.path.isdir(PROJECT_DIR):
        raise RuntimeError(
            f'git clone failed (exit {result.returncode}). Check REPO_URL and the output above.'
        )
else:
    assert os.path.exists(PROJECT_DIR), 'Upload the project to ' + PROJECT_DIR

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working dir:', os.getcwd())
print('Contents:', os.listdir('.'))

## 3. Install dependencies

In [ ]:
# torch/torchvision are preinstalled on Colab; install the rest quietly.
!pip install -q -r requirements.txt

## 4. Authenticate to Kaggle & download the dataset

**Easiest, works everywhere (incl. the VS Code <-> Colab bridge):** paste your
credentials into the cell below and run it. The `files.upload()` widget further
down often fails over the VS Code bridge, so prefer this.

Other supported sources (auto-detected by the auth cell):
Colab secrets (`KAGGLE_USERNAME`/`KAGGLE_KEY`), a `kaggle.json` in the project
directory, or the upload prompt.

> **Do NOT commit your real key.** This repo is public - keep the values below
> as placeholders in git and only fill them in locally.

In [ ]:
# >>> Paste your Kaggle credentials here, run this cell, then run the next one. <<<
# Find them in kaggle.json (Kaggle -> Account -> Create New API Token).
# Leave as placeholders when committing - do NOT push your real key to a public repo.
import os
os.environ['KAGGLE_USERNAME'] = 'your_username'  # <-- edit locally, don't commit
os.environ['KAGGLE_KEY'] = 'your_key'            # <-- edit locally, don't commit
print('Kaggle env vars set:', bool(os.environ.get('KAGGLE_USERNAME')) and bool(os.environ.get('KAGGLE_KEY')))

In [ ]:
import os, json, shutil

def _install_kaggle_json(src_path):
    """Copy a kaggle.json into place and export its creds as env vars."""
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy(src_path, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    with open(src_path) as f:
        creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']

# Look for a kaggle.json in the cwd or the project directory.
candidates = [os.path.join(os.getcwd(), 'kaggle.json')]
if 'PROJECT_DIR' in globals():
    candidates.append(os.path.join(PROJECT_DIR, 'kaggle.json'))
local_json = next((p for p in candidates if os.path.exists(p)), None)

if os.environ.get('KAGGLE_USERNAME') not in (None, '', 'your_username') and \
        os.environ.get('KAGGLE_KEY') not in (None, '', 'your_key'):
    print('Using KAGGLE_USERNAME / KAGGLE_KEY from environment / the cell above.')
elif local_json is not None:
    _install_kaggle_json(local_json)
    print('Using kaggle.json found at', local_json)
else:
    from google.colab import files
    print('No credentials found - upload your kaggle.json ...')
    uploaded = files.upload()
    with open('kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
    _install_kaggle_json('kaggle.json')
    print('Kaggle credentials installed from uploaded kaggle.json.')

In [ ]:
import kagglehub

# Downloads (and caches) the dataset onto the Colab VM, returns the local path.
DATA_DIR = kagglehub.dataset_download('pacificrm/skindiseasedataset')
print('Path to dataset files:', DATA_DIR)

import os
print('Top-level contents:', os.listdir(DATA_DIR))

### Inspect the extracted layout
The data module auto-detects `train/`+`test/` splits or a flat per-class folder, but it's worth a look.

In [ ]:
import os
for root, dirs, fnames in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + f'/   ({len(fnames)} files)')
    if depth > 2:
        dirs[:] = []

## 5. Configure & train
Load the default config, point it at the downloaded data, and tweak anything you like.

In [ ]:
from src.utils import load_config, seed_everything

cfg = load_config('configs/default.yaml')
cfg.data.data_dir = DATA_DIR
cfg.data.num_workers = 2

# --- tweak training here ---
cfg.training.epochs = 25
cfg.data.batch_size = 32
cfg.model.backbone = 'efficientnet_b0'  # try resnet50, efficientnet_b3, ...
cfg.output.checkpoint_dir = '/content/outputs/checkpoints'
cfg.output.log_dir = '/content/outputs/logs'

seed_everything(cfg.seed)
print(cfg)

In [ ]:
from src.data import SkinDiseaseDataModule

dm = SkinDiseaseDataModule(
    data_dir=cfg.data.data_dir,
    image_size=cfg.data.image_size,
    batch_size=cfg.data.batch_size,
    num_workers=cfg.data.num_workers,
    val_split=cfg.data.val_split,
    test_split=cfg.data.test_split,
    seed=cfg.seed,
)
dm.setup()
print(f'{dm.num_classes} classes detected:')
for i, name in enumerate(dm.class_names):
    print(f'  {i:2d}  {name}')

In [ ]:
from src.models import build_model
from src.training import Trainer

model = build_model(
    backbone=cfg.model.backbone,
    num_classes=dm.num_classes,
    pretrained=cfg.model.pretrained,
    dropout=cfg.model.dropout,
    freeze_backbone=cfg.model.freeze_backbone,
)

weights = dm.class_weights() if cfg.training.use_class_weights else None
trainer = Trainer(model, cfg, dm.class_names, class_weights=weights)
print('Device:', trainer.device, '| AMP:', trainer.use_amp)

In [ ]:
history = trainer.fit(dm.train_dataloader(), dm.val_dataloader())

### Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(history['train_acc'], label='train')
ax2.plot(history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('epoch'); ax2.legend()
plt.show()

## 6. Evaluate on the test split
Reloads the best checkpoint (highest val accuracy) before evaluating.

In [ ]:
from pathlib import Path
from src.training import classification_metrics, plot_confusion_matrix

trainer.load_checkpoint(Path(cfg.output.checkpoint_dir) / 'best.pth')
y_true, y_pred = trainer.predict(dm.test_dataloader())

metrics = classification_metrics(y_true, y_pred, dm.class_names)
print(f"Test accuracy : {metrics['accuracy']:.4f}")
print(f"Macro F1      : {metrics['macro_f1']:.4f}")
print(f"Weighted F1   : {metrics['weighted_f1']:.4f}")
print()
print(metrics['report_text'])

In [ ]:
fig = plot_confusion_matrix(
    y_true, y_pred, dm.class_names,
    save_path=Path(cfg.output.log_dir) / 'confusion_matrix.png',
)
plt.show()

## 7. Save the model to Google Drive
Persists the checkpoint so it survives after the Colab session ends.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/skin-disease-model'
os.makedirs(DEST, exist_ok=True)

shutil.copy(Path(cfg.output.checkpoint_dir) / 'best.pth', DEST)
shutil.copy(Path(cfg.output.log_dir) / 'confusion_matrix.png', DEST)
shutil.copy(Path(cfg.output.log_dir) / 'history.json', DEST)
print('Saved to', DEST)
print(os.listdir(DEST))

## 8. (Optional) Single-image prediction

In [ ]:
from src.inference import SkinDiseasePredictor

predictor = SkinDiseasePredictor(Path(cfg.output.checkpoint_dir) / 'best.pth')
from google.colab import files
up = files.upload()
img_path = list(up.keys())[0]
for r in predictor.predict(img_path, top_k=5):
    print(f"{r['label']:<25} {r['probability']:.3f}")